# 가벼운 주소 DB 사용
완성된 작은 카탈로그 DB가 들어 있습니다. 원본 파르케 3개가 있는 폴더만 지정하면 됩니다. 원본 전체를 DB에 복사하지 않습니다.

In [ ]:
# 필요한 경우 실행
# %pip install duckdb pandas
from pathlib import Path
from IPython.display import display
from address_lite import LiteAddressDB, build_catalog

CATALOG_PATH=Path("address_catalog.duckdb")
# 본인 컴퓨터에서 원본 파르케 3개가 있는 폴더 경로만 수정하세요.
SOURCE_DIR=Path("/Users/chie/실제_파르케_폴더")
engine=LiteAddressDB(CATALOG_PATH,SOURCE_DIR,verify_hash=True)
display(engine.con.sql("SELECT * FROM sources ORDER BY 기준연도").df())

In [ ]:
# 기존 코드·이름 유지 + 시/구 분리 목록
display(engine.find_regions(sido='경기도',city='고양시',district='덕양구'))
display(engine.con.sql("SELECT * FROM sigungu_hierarchy_links WHERE city_name='고양시' ORDER BY 기준연도,district_name").df())
display(engine.con.sql('SELECT * FROM road_table ORDER BY 기준연도,road_id LIMIT 20').df())

In [ ]:
# 실제 원본 주소 한 건으로 후보 조회. 인위적인 주소를 생성하지 않습니다.
sample=engine.con.execute('SELECT 법정동시도명,법정동시군구명,법정동읍면동명,법정동리명,지번본번,지번부번 FROM read_parquet(?) WHERE 일련번호=1',[str(engine.sources[2025])]).fetchall()[0]
result=engine.lot_by_region(sido=sample[0],sigungu=sample[1],emd=sample[2],ri=sample[3],main=int(sample[4]),sub=int(sample[5]))
print('후보 상태:',result['status'])
print('비교 코드:',result['comparison_code_fields'])
print('입력 주소 작성연도:',result['input_address_year'])
display(result['code_combinations'])
for item in result['per_search']:
    if item['candidate_count']:
        print(item['reference_year'],item['candidate_count'],item['search_conditions'])
        display(item['candidates'].head())

## 작은 DB 재생성 — 원본을 교체했을 때만
기존 DB를 덮어쓰지 않도록 새 경로에 생성합니다. 보통은 실행할 필요가 없습니다.

In [ ]:
engine.close()
REBUILD=False
if REBUILD:
    print(build_catalog(SOURCE_DIR, Path('address_catalog_rebuilt.duckdb')))